# 🗺️ Lab 2.5: The Meaning Machine — Word Embeddings & Semantic Search

In Lab 1 you proved Bag of Words can't see meaning. Now meet the fix: **embeddings** — every word (or sentence) becomes a point on a giant meaning-map, where neighbors mean similar things.

## 📚 The plan
1. 👑 Play with a real word map: `king − man + woman = ?`
2. 🏦 Expose the **bank problem** (one word, one vector, two meanings)
3. 🧠 Watch **contextual embeddings** solve it live
4. 🔎 Build your own **semantic search engine**

⏳ First run downloads two small models (~150 MB total) — start the setup cell early!

In [1]:
!pip install pandas numpy gensim sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 64.5 MB/s eta 0:00:00


### 🛠️ Setup (start this NOW — it downloads models)

In [2]:
!pip install gensim sentence-transformers -q

import numpy as np
import pandas as pd
import gensim.downloader as api

# Classic static word embeddings: GloVe, trained on 6 billion words of Wikipedia + news
print("⏳ Downloading the word map (~66 MB, one-time)...")
glove = api.load('glove-wiki-gigaword-50')
print(f"🗺️ Map loaded! {len(glove.key_to_index):,} words, each a vector of {glove.vector_size} numbers.")

⏳ Downloading the word map (~66 MB, one-time)...
[==================================================] 100.0% 66.0/66.0MB downloaded
🗺️ Map loaded! 400,000 words, each a vector of 50 numbers.


### 👑 Section 1: Explore the Meaning Map

Every word is now a list of 50 numbers — its **coordinates in meaning-space**. Words that appear in similar contexts got pushed close together during training. Let's verify.

In [3]:
print("The vector for 'king' (first 10 of 50 numbers):")
print(glove['king'][:10].round(3))
print("\nMeaningless to us — but DISTANCES between vectors are meaningful. Watch:")

The vector for 'king' (first 10 of 50 numbers):
[ 0.505  0.686 -0.595 -0.023  0.6   -0.135 -0.088  0.474 -0.618 -0.31 ]

Meaningless to us — but DISTANCES between vectors are meaningful. Watch:


In [4]:
for word in ['coffee', 'university', 'happy']:
    neighbors = glove.most_similar(word, topn=5)
    print(f"🏘️ Neighbors of '{word}':  " + ", ".join(f"{w} ({s:.2f})" for w, s in neighbors))

🏘️ Neighbors of 'coffee':  drink (0.82), drinks (0.82), wine (0.81), tea (0.81), beer (0.80)
🏘️ Neighbors of 'university':  college (0.87), harvard (0.87), yale (0.86), graduate (0.86), institute (0.85)
🏘️ Neighbors of 'happy':  'm (0.91), everyone (0.90), everybody (0.90), really (0.88), me (0.88)


In [5]:
# 🎩 The most famous trick in NLP: vector arithmetic on MEANING
result = glove.most_similar(positive=['king', 'woman'], negative=['man'], topn=3)
print("👑 king − man + woman = ?")
for word, score in result:
    print(f"   {word}  ({score:.2f})")

# The map learned the CONCEPT of gender as a direction in space. Nobody programmed this.

👑 king − man + woman = ?
   queen  (0.85)
   throne  (0.77)
   prince  (0.76)


### 🎯 Mini Exercise 1.1 — Analogy Machine
Try your own analogies with the same pattern `A − B + C = ?`:
- `paris − france + japan = ?` (capital-of direction 🗾)
- `walked − walk + swim = ?` (past-tense direction 🏊)
- Invent one — does the map know it?

In [6]:
# TO-DO: your analogy. Pattern: positive=[A, C], negative=[B]
glove.most_similar(positive=['paris', 'japan'], negative=['france'], topn=3)

[('tokyo', 0.9234451055526733),
 ('shanghai', 0.8370239734649658),
 ('japanese', 0.7904196977615356)]

In [7]:
# One more toy: odd one out 🕵️
print(glove.doesnt_match(['breakfast', 'lunch', 'dinner', 'laptop']))
print(glove.doesnt_match(['saudi', 'egypt', 'jordan', 'banana']))

laptop
banana


### 🏦 Section 2: The Bank Problem — Static Embeddings Hit a Wall

Every word in GloVe has exactly **ONE vector, forever**. But read these:

1. *"I deposited money in the **bank**."* 💰
2. *"We had a picnic on the river **bank**."* 🌊

Same word, two meanings. GloVe is forced to give both the SAME vector — a weird average of money-bank and river-bank. Look at its neighbors:

In [8]:
print("🏦 Neighbors of 'bank':")
for w, s in glove.most_similar('bank', topn=8):
    print(f"   {w} ({s:.2f})")

# 👀 It's ALL finance. The river meaning got buried. One vector cannot hold two meanings.

🏦 Neighbors of 'bank':
   banks (0.87)
   securities (0.80)
   banking (0.80)
   investment (0.78)
   exchange (0.78)
   financial (0.77)
   credit (0.76)
   lender (0.75)


### 🧠 Section 3: Contextual Embeddings — The Vector Depends on the Sentence

Modern models (BERT and friends) compute the vector **fresh from the surrounding sentence**. We'll use a sentence-transformer: it reads a whole sentence and produces one meaning-vector for it — with attention deciding what matters.

To compare vectors we use **cosine similarity**: 1.0 = same meaning, near 0 = unrelated.

In [9]:
from sentence_transformers import SentenceTransformer, util

print("⏳ Downloading the contextual model (~90 MB, one-time)...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("🧠 Contextual model ready!")

⏳ Downloading the contextual model (~90 MB, one-time)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🧠 Contextual model ready!


In [10]:
sentences = [
    "I withdrawn money from the bank",          # 💰 money-bank
    "We had a picnic on the river bank",      # 🌊 river-bank
    "The financial institution approved my loan",   # 💰 pure finance, no 'bank'
    "We sat by the stream under the trees",          # 🌊 pure nature, no 'bank'
]

vecs = model.encode(sentences)
sim = util.cos_sim(vecs, vecs).numpy().round(2)

pd.DataFrame(sim, index=['💰 money bank', '🌊 river bank', '💰 loan (no "bank")', '🌊 stream (no "bank")'],
                  columns=['💰 money bank', '🌊 river bank', '💰 loan', '🌊 stream'])

,💰 money bank,🌊 river bank,💰 loan,🌊 stream
💰 money bank,1.00,0.32,0.46,0.09
🌊 river bank,0.32,1.00,0.16,0.45
"💰 loan (no ""bank"")",0.46,0.16,1.00,0.05
"🌊 stream (no ""bank"")",0.09,0.45,0.05,1.00


**Read the matrix carefully — this is the punchline of the day:**
- 💰 *money bank* is **more similar to the loan sentence** (which never says "bank") than to 🌊 *river bank* (which does!)
- 🌊 *river bank* pairs up with the stream sentence.

The model matched **meanings, not words**. The word "bank" got a different effective vector in each sentence — attention read the context and adjusted. This is exactly what Bag of Words (Lab 1) and static GloVe (Section 2) could never do. 🤯

### 🔎 Section 4: Build a Semantic Search Engine (20 lines!)

Search that matches **meaning instead of keywords** — the same technique behind modern search, recommendations, and (spoiler for tomorrow 👀) RAG. The recipe:
1. Embed all your documents → vectors
2. Embed the query → vector
3. Return the documents with highest cosine similarity

In [11]:
# Our tiny 'document collection': one line per topic from this week!
docs = [
    "Supervised learning trains models on labeled examples like spam or not spam",
    "K-Means groups customers into clusters without any labels",
    "Convolutional neural networks detect edges and shapes in images",
    "Data augmentation flips and modifies photos to create free training data",
    "Transfer learning reuses a pretrained network for a new task",
    "Overfitting means the model memorizes training data instead of learning",
    "Precision and recall measure different kinds of classification mistakes",
    "Word embeddings place similar words close together in vector space",
    "The mall dataset was segmented by income and spending score",
    "Mushroom classification predicted whether a mushroom is poisonous",
]

doc_vecs = model.encode(docs)

def search(query, top_k=3):
    # convert the query string into the same embedding space as the documents
    # model.encode expects a list, so we wrap query in [query] even though it's one string
    q_vec = model.encode([query])

    # compute cosine similarity between the query vector and every document vector
    # cos_sim returns a matrix, so [0] pulls out the single row of scores (one per doc)
    scores = util.cos_sim(q_vec, doc_vecs)[0]

    # sort scores from highest to lowest similarity, then keep only the top_k indices
    best = scores.argsort(descending=True)[:top_k]

    print(f"🔎 Query: '{query}'\n")
    for rank, idx in enumerate(best, 1):
        print(f"  {rank}. ({scores[idx]:.2f}) {docs[idx]}")



search("how do I stop my model from memorizing?")

🔎 Query: 'how do I stop my model from memorizing?'

  1. (0.53) Overfitting means the model memorizes training data instead of learning
  2. (0.34) Precision and recall measure different kinds of classification mistakes
  3. (0.26) Supervised learning trains models on labeled examples like spam or not spam


In [12]:
# The magic: ZERO shared keywords, correct result anyway
search("grouping shoppers by behavior")   # should find K-Means / mall — no shared words!

🔎 Query: 'grouping shoppers by behavior'

  1. (0.49) K-Means groups customers into clusters without any labels
  2. (0.36) The mall dataset was segmented by income and spending score
  3. (0.21) Supervised learning trains models on labeled examples like spam or not spam


### 🎯 Mini Exercise 4.1
1. Ask three questions of your own — try phrasing them with words that appear NOWHERE in the documents. Does it still find the right one?
2. Add 3 new documents about topics you like (games, football, cooking...) and search across them. 🎮⚽🍳

In [13]:
# TO-DO: your searches and your documents
search("")
search("")
search("")

🔎 Query: ''

  1. (0.13) K-Means groups customers into clusters without any labels
  2. (0.12) Data augmentation flips and modifies photos to create free training data
  3. (0.10) Supervised learning trains models on labeled examples like spam or not spam
🔎 Query: ''

  1. (0.13) K-Means groups customers into clusters without any labels
  2. (0.12) Data augmentation flips and modifies photos to create free training data
  3. (0.10) Supervised learning trains models on labeled examples like spam or not spam
🔎 Query: ''

  1. (0.13) K-Means groups customers into clusters without any labels
  2. (0.12) Data augmentation flips and modifies photos to create free training data
  3. (0.10) Supervised learning trains models on labeled examples like spam or not spam


## The GOLDEN Question 🏆

Your search engine matched *"grouping shoppers by behavior"* to the K-Means document - **zero words in common**.

**Explain HOW, using the two big ideas of this lab (meaning-space + context). Then think ahead: tomorrow we'll want an AI assistant to answer questions using OUR documents. Which part of what you just built would be the perfect first step?** 🤔